# Mission Analysis

Post-mission analysis notebook for ISR-RL-DMPC scenario results.
Loads JSON output files produced by `scripts/run_dmpc.py` and
`scripts/run_dmpc_rl.py` and visualises per-episode metrics.

**Produce results first:**
```bash
# Pure DMPC
python scripts/run_dmpc.py --scenario area_surveillance --episodes 5
python scripts/run_dmpc.py --scenario threat_response   --episodes 5
python scripts/run_dmpc.py --scenario search_and_track  --episodes 5

# DMPC-RL (requires trained model)
python scripts/run_dmpc_rl.py --scenario area_surveillance --episodes 5
python scripts/run_dmpc_rl.py --scenario threat_response   --episodes 5
python scripts/run_dmpc_rl.py --scenario search_and_track  --episodes 5
```

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.figsize': (13, 5), 'font.size': 11})

RESULTS_ROOT = Path('../data/results')
DMPC_DIR    = RESULTS_ROOT / 'dmpc'
DMPC_RL_DIR = RESULTS_ROOT / 'dmpc_rl'

SCENARIOS = ['area_surveillance', 'threat_response', 'search_and_track']

def load_results(base_dir: Path, method: str) -> dict[str, list[dict]]:
    """Load all JSON result files for a method, grouped by scenario."""
    results: dict[str, list[dict]] = {s: [] for s in SCENARIOS}
    if not base_dir.exists():
        return results
    for path in sorted(base_dir.glob('*.json')):
        with open(path) as f:
            episodes = json.load(f)
        for ep in (episodes if isinstance(episodes, list) else [episodes]):
            sc = ep.get('scenario', path.stem.split('_2')[0])
            if sc in results:
                results[sc].append(ep)
    total = sum(len(v) for v in results.values())
    print(f'{method}: {total} episodes loaded from {base_dir}')
    return results

dmpc_results    = load_results(DMPC_DIR,    'Pure DMPC')
dmpc_rl_results = load_results(DMPC_RL_DIR, 'DMPC-RL')

In [ ]:
def episode_metric(results: dict, scenario: str, key: str) -> np.ndarray:
    eps = results.get(scenario, [])
    vals = [ep.get(key, np.nan) for ep in eps]
    return np.array(vals, dtype=float)

def bar_comparison(metric: str, ylabel: str, title: str, ax):
    x = np.arange(len(SCENARIOS))
    w = 0.35
    dmpc_means = [np.nanmean(episode_metric(dmpc_results, s, metric)) for s in SCENARIOS]
    rl_means   = [np.nanmean(episode_metric(dmpc_rl_results, s, metric)) for s in SCENARIOS]
    dmpc_stds  = [np.nanstd(episode_metric(dmpc_results, s, metric)) for s in SCENARIOS]
    rl_stds    = [np.nanstd(episode_metric(dmpc_rl_results, s, metric)) for s in SCENARIOS]

    bars1 = ax.bar(x - w/2, dmpc_means, w, yerr=dmpc_stds,
                   label='Pure DMPC', color='steelblue', capsize=4, alpha=0.85)
    bars2 = ax.bar(x + w/2, rl_means,   w, yerr=rl_stds,
                   label='DMPC-RL',  color='tomato',    capsize=4, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('_', '\n') for s in SCENARIOS], fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
bar_comparison('total_reward',       'Total reward',    'Total Reward per Scenario',        axes[0])
bar_comparison('episode_steps',      'Steps completed', 'Steps Completed per Scenario',     axes[1])
bar_comparison('final_battery_mean', 'Battery (0–1)',   'Final Battery Level per Scenario', axes[2])
plt.tight_layout()
plt.savefig('mission_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved mission_analysis.png')

In [ ]:
# ── Per-scenario reward distribution ───────────────────────────────
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(15, 5))
for ax, sc in zip(axes, SCENARIOS):
    d  = episode_metric(dmpc_results,    sc, 'total_reward')
    rl = episode_metric(dmpc_rl_results,  sc, 'total_reward')
    data, labels, colors = [], [], []
    if len(d[~np.isnan(d)])  > 0: data.append(d[~np.isnan(d)]);  labels.append('Pure DMPC'); colors.append('steelblue')
    if len(rl[~np.isnan(rl)]) > 0: data.append(rl[~np.isnan(rl)]); labels.append('DMPC-RL');   colors.append('tomato')
    if data:
        bp = ax.boxplot(data, labels=labels, patch_artist=True, notch=False)
        for patch, c in zip(bp['boxes'], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.7)
    ax.set_title(sc.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Total reward')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Reward Distribution per Scenario', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Solve-time comparison ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bar_comparison('mean_solve_time_ms', 'Solve time (ms)', 'Mean DMPC Solve Time per Scenario', ax)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────
hdr = '{:<22} {:<12} {:>8} {:>10} {:>10} {:>8} {:>10}'.format(
    'Scenario', 'Method', 'Episodes', 'Reward μ', 'Reward σ', 'Steps μ', 'Battery μ')
print(hdr)
print('-' * 85)
for sc in SCENARIOS:
    for label, res in [('Pure DMPC', dmpc_results), ('DMPC-RL', dmpc_rl_results)]:
        r = episode_metric(res, sc, 'total_reward')
        s = episode_metric(res, sc, 'episode_steps')
        b = episode_metric(res, sc, 'final_battery_mean')
        n = int(np.sum(~np.isnan(r)))
        if n == 0:
            print('{:<22} {:<12} {:>8}'.format(sc, label, '—'))
        else:
            print('{:<22} {:<12} {:>8} {:>10.2f} {:>10.2f} {:>8.0f} {:>10.3f}'.format(
                sc, label, n, np.nanmean(r), np.nanstd(r),
                np.nanmean(s), np.nanmean(b)))